# Week 35, Day 2: Spike-triggered covariance: finding more than one filter

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Recovering two filters

A model complex cell with quadrature filters k1 and k2 is driven by white noise. Compute the STA (weighted by spike counts), the STC (the weighted covariance of spike-triggered stimuli about the STA), subtract the stimulus covariance, and take eigenvalues `ev` (ascending) and eigenvectors. Measure the `overlap` of the top two eigenvectors with the plane of k1 and k2 (1 is perfect).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2006)
D, T = 20, 20000
X = np.lib.stride_tricks.sliding_window_view(r.normal(size=T + D - 1), D)[:T][:, ::-1]
lag = np.arange(D)
k1 = np.exp(-lag / 5) * np.sin(lag / 2); k1 /= np.linalg.norm(k1)
k2 = np.exp(-lag / 5) * np.cos(lag / 2); k2 -= (k2 @ k1) * k1; k2 /= np.linalg.norm(k2)
y = r.poisson(0.05 * ((X @ k1) ** 2 + (X @ k2) ** 2))

sta = ...
Xc = X - sta
stc = ...                                       # spike-weighted covariance about the STA
ev, V = np.linalg.eigh(stc - np.cov(X.T))
B = V[:, -2:]
overlap = np.linalg.norm(B.T @ np.c_[k1, k2]) ** 2 / 2
print(f"{y.sum()} spikes; STA norm {np.linalg.norm(sta):.3f}")
print("largest eigenvalues:", np.round(ev[-4:], 3), " smallest:", np.round(ev[:2], 3))
print(f"overlap of the top two eigenvectors with the filters' plane: {overlap:.3f}")
plt.plot(ev, "o"); plt.xlabel("rank"); plt.ylabel("STC eigenvalue minus prior"); plt.show()

In [ ]:
import numpy as np
def test_sta():
    "the STA is small: its norm is below 0.3"
    assert np.linalg.norm(sta) < 0.3

def test_stc():
    "the two largest eigenvalues stand well above the rest, and their eigenvectors span the filters' plane (overlap above 0.9)"
    assert ev[-2] > 4 * abs(ev[-3]) and overlap > 0.9

test_sta()
test_stc()
print("All checks passed")

### Exercise: A shuffle null for eigenvalues

For 20 random circular shifts of y (by at least 100 bins), recompute the STC eigenvalues and record their maximum and minimum. Set `band` to (lowest minimum, highest maximum) across shifts and count how many real eigenvalues lie above it.

In [ ]:
def stc_eigs(yy):
    s = X.T @ yy / yy.sum(); Xs = X - s
    return np.linalg.eigvalsh((Xs * yy[:, None]).T @ Xs / yy.sum() - np.cov(X.T))

mins, maxs = [], []
for _ in range(20):
    e = stc_eigs(np.roll(y, r.integers(100, T - 100)))
    ...
band = (min(mins), max(maxs))
print(f"null band {band[0]:.3f} to {band[1]:.3f};  {np.sum(ev > band[1])} eigenvalues above it, {np.sum(ev < band[0])} below")

In [ ]:
import numpy as np
def test_band():
    "exactly two real eigenvalues exceed the top of the null band"
    assert np.sum(ev > band[1]) == 2

test_band()
print("All checks passed")